# Chain-length comparison — the periodic random slab at 5, 10, 15, 20 and 30 beads per strand

How the gel's observables scale with the strand length.  Two inputs per N (2026-11-03):

* the **equilibration** of each slab (`simulations/chain_length_testing`, `slab_with_support.lmp` with the Rg-plateau halt:
  production ends once the block-mean $R_g$ has stopped moving) — $R_g(t)$, thickness and in-plane box overlaid, with the
  step at which each halt fired;
* the **compression** of each equilibrated slab (`triaxial_compression_two_pist` on the two-piston conversion of its
  `final_config`, one level, no unload — so M, G and $\varphi_s$ but no $D_c$), each loaded with `lib/triaxial.py` exactly as
  `triaxial_compression_single_two_pist.ipynb` loads it (`compare.load_deck`) and then drawn **against N** with a fitted
  power law per observable.  The comparison code is `lib/chain_length.py`.

The 5-bead point is the existing random periodic slab (`slab_support_periodic_5beads_tall_rho04_random`, 14 M steps,
2026-11-02) and its compression run; the other four come from the sweep (input files `scripts/lib/build_beads_sweep.py`,
`slab_data_file_info.md`).  `CFG` below must name each N's compression DATANAME / RUN_ID / level once those runs exist;
an N whose files are missing is skipped with a note, so the notebook runs while the sweep is still in flight.


## 1 · Setup

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: triaxial.py (compression analysis), compare.py (deck loader), chain_length.py (this notebook)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
import compare as cmp
import chain_length as cl
tri = importlib.reload(tri); cmp = importlib.reload(cmp); cl = importlib.reload(cl)
tri.setup_style()
print('comparison code: ', LIB / 'chain_length.py')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit
# ══════════════════════════════════════════════════════════════════════════
NBEADS = [5, 10, 15, 20, 30]           # the sweep (cl.SWEEP: equilibration DATANAME + step budget per N)
INTERACTION = "1.0_1.0"
COMP_LEVEL = "0.10"                    # the one compression level of every N (triaxial_compression_two_pist_random.batch: eps 0.10, 9 M hold)

# Where the EQUILIBRATION traces are read from (first hit wins): the run directories on Expanse
# (~/Documents/lammps_runs/<folder>/...) when this runs on the login node, or a local copy of each run's
# output_files/volume_data/*.dat + log.lammps under flow_data_local/chain_length/<run dir name>/ (scp them;
# cl.load_equilibration also reads the bare .dat files dropped straight into that folder).
EQ_ROOTS = ('~/Documents/lammps_runs/chain_length_testing', '~/Documents/lammps_runs/slab_with_support',
            '../../flow_data_local/chain_length')

# One compression Config per N -- the Config cell of triaxial_compression_single_two_pist.ipynb for that run.
# DATANAME = the two-piston converter's output stem (cl.eq_final_config(N) + '_two_pist' when the converter ran with
# its defaults); RUN_ID = the local folder under flow_data_local/{compression,plots}.  Edit as the runs land.
CFG = {}
for N in NBEADS:
    CFG[N] = tri.Config(
        DATANAME    = cl.eq_final_config(N, INTERACTION) + "_two_pist",
        INTERACTION = INTERACTION,
        NSTEPS      = None,                        # resolves each level's auto-sized hold tag from the files
        RUN_ID      = f"chain_length_{N}beads_comp_{COMP_LEVEL}",
        COMP_LEVELS = [COMP_LEVEL],
        mode        = "compression",
        two_pist    = True,
        plateau_frac      = 0.25,
        plateau_frac_auto = 0.45,
        VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, P_CAL = 1.5,
        M_SUBTRACT_REF = True, G_SUBTRACT_REF = True,
    )
# the 5-bead compression already ran under its own name (chain_random_periodic.batch, 2026-11-02)
CFG[5].DATANAME = "final_config_slab_support_periodic_5beads_tall_rho04_random_1.0_1.0_14000000_two_pist"


In [ ]:
SYNC, FORCE_SYNC = True, False       # pull each N's compression files from Expanse (one login); skipped for an N with no run yet
if SYNC:
    for N, c in CFG.items():
        try:
            cmp.sync_deck(c, force=FORCE_SYNC)
        except Exception as err:
            print(f'N = {N}: sync skipped ({err})')


## 2 · Equilibration: when did each slab stop swelling?

In [ ]:
EQ = [cl.load_equilibration(N, roots=EQ_ROOTS, interaction=INTERACTION) for N in NBEADS]
cl.fig_equilibration(EQ, out=Path('../../flow_data_local/plots/chain_length_equilibration.png')
                     if Path('../../flow_data_local/plots').exists() else None);


## 3 · Compression observables against N

In [ ]:
# Each N loaded by its own library (M, G, phi per level: the single-level notebook's numbers); an N whose files
# are not there yet is skipped.  Voronoi phi profiles come from the disk cache the source notebooks fill.
DECKS = {}
for N, c in CFG.items():
    try:
        DECKS[N] = cmp.load_deck(c, verbose=False)
    except Exception as err:
        print(f'N = {N}: not loaded ({type(err).__name__}: {err})')
        DECKS[N] = None
ROWS = cl.collect_observables(DECKS, EQ)
cl.print_table(ROWS)


In [ ]:
# M (network and load piston), G (lateral network stress), the interior solvent fraction and the polymer fraction,
# each vs N on log axes with a fitted power law; dotted: the c*-theorem guides (phi_p ~ N^-4/5, G ~ N^-9/5) pinned to the
# first point.  The 1/N of an ideal network at fixed volume is what M, G would follow WITHOUT the re-swelling.
cl.fig_observables_vs_N(ROWS, out=Path('../../flow_data_local/plots/chain_length_observables.png')
                        if Path('../../flow_data_local/plots').exists() else None);


## Notes

* **Why this sweep (2026-11-03).**  At 5 beads per strand the gel is elastic rather than osmotic: the strands are a few
  bonds long, so the network's elasticity rather than the mixing term sets the swelling.  The sweep keeps the slab's
  physical size (~43 σ in plane, ~111 σ thick, 180–256 k atoms) and changes only the mesh: the input files were built at
  the lattice constant the gel is *expected* to swell to, $a_{eq}(N) = 4.82\,((N-1)/4)^{0.59}$ (the equilibrated
  5-bead random slab scaled with the good-solvent exponent), so the aniso barostat has little to correct.  The
  equilibration panel checks that: $l_x$ should move by a few percent, not tens.
* **Where the equilibration ends.**  The deck's Rg-plateau halt (RG_BLOCK-step blocks, two consecutive block means
  within 0.15 %) fires at the star in the $R_g$ panel; a slab that ran to its step budget without a star has not
  plateaued by that test and its compression run inherits whatever drift is left.
* **Scaling guides.**  For an ideal network at fixed volume $G \propto \nu k_BT$ with $\nu \propto 1/N$.  In good-solvent
  swelling equilibrium the c* theorem gives $\varphi_p \propto N^{-4/5}$ and $G \propto N^{-9/5}$ (de Gennes); M carries
  the osmotic modulus $K_{os} = \varphi\,\partial\Pi/\partial\varphi \sim \varphi^{9/4}$ on top of $4G/3$.  The fitted
  exponents are printed in the legends.
* **No $D_c$ here** — the compression runs of the sweep have no unload (the trusted $D_c$ is the unload's).  Add
  `UNLOAD=1` levels later and the cross-deck notebook picks them up.
